<a href="https://colab.research.google.com/github/lab30041954/ML_Course/blob/main/Assignments/Assignment%201%20-%20The%20prof%20solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2 - Yang Cao

## Question 2

### Task

Change the features matrix by: (a) dropping the three cap_ features and (b) binarizing all the word_ features, transforming every column into a dummy for the occurrence of the corresponding word, taking value 1 if the word occurs in the message and 0 otherwise. Based on this new features matrix, train two new spam filters, one based on a logistic regression model and the other one based on a decision tree model, using the binarized data set. Evaluate these new filters and compare them to those obtained before.

### Preparation

We first import the dataset, inspect the first few observations, and compute the proportion of spam messages in the sample, which agrees with the description (1,813 out of 4,601 emails).

In [2]:
import pandas as pd
path = 'https://raw.githubusercontent.com/lab30041954/Data/main/'
df = pd.read_csv(path + 'spam.csv')
df.head()

,word_make,word_address,word_all,word_3d,word_our,word_over,word_remove,word_internet,word_order,word_mail,...,word_original,word_project,word_re,word_edu,word_table,word_conference,cap_ave,cap_long,cap_total,spam
0,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,...,0.00,0.0,0.00,0.00,0.0,0.0,3.756,61,278,1
1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,...,0.00,0.0,0.00,0.00,0.0,0.0,5.114,101,1028,1
2,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,...,0.12,0.0,0.06,0.06,0.0,0.0,9.821,485,2259,1
3,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.0,0.00,0.00,0.0,0.0,3.537,40,191,1
4,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,...,0.00,0.0,0.00,0.00,0.0,0.0,3.537,40,191,1


In [3]:
round(df['spam'].mean(), 3)

np.float64(0.394)

We create a target vector and a features matrix.

In [4]:
y = df['spam']
X = df.drop(columns='spam')

### Construct a modified feature matrix

(a) dropping the three cap_ features

In [5]:
X_new = X.drop(columns=[c for c in X.columns if c.startswith('cap_')])

(b) binarizing all the word_ features, transforming every column into a dummy for the occurrence of the corresponding word, taking value 1 if the word occurs in the message and 0 otherwise

In [6]:
word_cols = [c for c in X_new.columns if c.startswith('word_')]
X_new[word_cols] = (X_new[word_cols] > 0).astype(int)

I inspect the first few rows of the modified feature matrix to verify that everything is correct.

In [7]:
X_new.head()

,word_make,word_address,word_all,word_3d,word_our,word_over,word_remove,word_internet,word_order,word_mail,...,word_pm,word_direct,word_cs,word_meeting,word_original,word_project,word_re,word_edu,word_table,word_conference
0,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,0,1,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1,0,1,0,1,1,1,1,1,1,...,0,1,0,0,1,0,1,1,0,0
3,0,0,0,0,1,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


### Logistic Regression

I import the logistic regression classifier from scikit-learn.

In [21]:
from sklearn.linear_model import LogisticRegression

#### Original feature matrix

I train a logistic regression model using the original feature matrix and evaluate its classification accuracy.

In [20]:
logit_orig = LogisticRegression(max_iter=5000)
logit_orig.fit(X, y)
round(logit_orig.score(X, y), 3)

0.92

#### Modified feature matrix

Then, I train a logistic regression model using the modified feature matrix.

In [22]:
logit_bin = LogisticRegression(max_iter=5000)
logit_bin.fit(X_new, y)
round(logit_bin.score(X_new, y), 3)

0.924

**The results show that the modified feature matrix slightly improves the classification accuracy of the logistic regression model.**

### Decision Tree (Modified Feature Matrix)

I train a decision tree classifier with **maximum depth four** on the modified feature matrix, since the same model is already evaluated on the original features and a depth of four provides the most appropriate trade-off between the true positive rate and the false positive rate.

In [23]:
from sklearn.tree import DecisionTreeClassifier
tree_bin4 = DecisionTreeClassifier(criterion='entropy', max_depth=4)
tree_bin4.fit(X_new, y)

DecisionTreeClassifier(criterion='entropy', max_depth=4)

I compute the confusion matrix.

In [24]:
from sklearn.metrics import confusion_matrix
y_pred_bin4 = tree_bin4.predict(X_new)
conf_bin4 = confusion_matrix(y, y_pred_bin4)
conf_bin4

array([[2645,  143],
       [ 409, 1404]])

I compute true positive rate (TPR) and false positive rate (FPR)

In [25]:
tpr_bin4 = conf_bin4[1, 1] / conf_bin4[1, :].sum()
fpr_bin4 = conf_bin4[0, 1] / conf_bin4[0, :].sum()

round(tpr_bin4, 3), round(fpr_bin4, 3)

(np.float64(0.774), np.float64(0.051))

**When using the modified feature matrix with binarized word variables, the decision tree classifier with maximum depth four exhibits a substantially lower false positive rate (0.051 compared to 0.109) than the model trained on the original features. However, this improvement comes at the cost of a reduced true positive rate (0.774 compared to 0.883). This result suggests that binarization makes the model more conservative, reducing the misclassification of legitimate emails while missing a larger fraction of spam messages.**